In [1]:
import torch
import torch.nn as nn
import torchvision.transforms as transforms
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cuda


In [ ]:
class HybridPolyAttnClassifier(nn.Module):
    def __init__(self, backbone_name='resnet18', pretrained=True, num_classes=5,
                 embed_dim=512, poly_proj_dim=256, num_chunks=8, num_heads=4, dropout=0.2):
        super().__init__()
        self.backbone = timm.create_model(backbone_name, pretrained=pretrained,
                                          num_classes=0, global_pool='avg')
        self.embed_dim = embed_dim or self.backbone.num_features
        self.poly = LowRankSecondOrder(in_dim=self.embed_dim, proj_dim=poly_proj_dim)
        poly_out_dim = self.poly.out_dim
        assert poly_out_dim % num_chunks == 0
        self.attn = AttentionOverChunks(embed_dim=poly_out_dim,
                                        num_chunks=num_chunks, num_heads=num_heads)
        final_dim = poly_out_dim // num_chunks
        self.classifier = nn.Sequential(
            nn.LayerNorm(final_dim),
            nn.Dropout(dropout),
            nn.Linear(final_dim, num_classes)
        )

In [3]:
model = HybridPolyAttnClassifier(
    backbone_name='resnet18',
    pretrained=False,
    num_classes=5,
    embed_dim=512,
    poly_proj_dim=256,
    num_chunks=8,
    num_heads=4
).to(device)

# Load weights
model.load_state_dict(torch.load("./models/best_hybrid_gray.pth", map_location=device))
model.eval()
print("Model loaded successfully!")

/home/parasite/.pyenv/versions/3.10.13/lib/python3.10/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/home/parasite/.pyenv/versions/3.10.13/lib/python3.10/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)
/tmp/ipykernel_8599/2894730437.py:12: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights

RuntimeError: Error(s) in loading state_dict for HybridPolyAttnClassifier:
	Missing key(s) in state_dict: "fc.0.weight", "fc.0.bias", "fc.2.weight", "fc.2.bias". 
	Unexpected key(s) in state_dict: "poly.V.weight", "poly.lin.weight", "poly.lin.bias", "attn.mha.in_proj_weight", "attn.mha.in_proj_bias", "attn.mha.out_proj.weight", "attn.mha.out_proj.bias", "attn.ff.0.weight", "attn.ff.0.bias", "attn.ff.2.weight", "attn.ff.2.bias", "classifier.0.weight", "classifier.0.bias", "classifier.2.weight", "classifier.2.bias". 

In [ ]:
infer_transform = transforms.Compose([
    transforms.Grayscale(num_output_channels=1),  # ensure grayscale
    transforms.Resize((48, 48)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5], std=[0.5]),
])

In [ ]:
class_names = ['Happy', 'Surprise', 'Sad', 'Fear', 'Angry']

def predict_outside_image(path, topk=3, show=True):
    img = Image.open(path)
    
    # Convert to grayscale explicitly
    gray_img = img.convert("L")
    tensor = infer_transform(gray_img).unsqueeze(0).to(device)

    print("Input shape:", tensor.shape, "dtype:", tensor.dtype,
          "min/max:", tensor.min().item(), tensor.max().item())

    with torch.no_grad():
        outputs, _ = model(tensor)
        probs = torch.softmax(outputs, dim=1)[0]
        topk_vals, topk_idx = torch.topk(probs, topk)

    if show:
        plt.imshow(gray_img, cmap='gray')
        plt.axis('off')
        plt.title("Grayscale Input Image")
        plt.show()

    print("Top predictions:")
    for p, i in zip(topk_vals.cpu().numpy(), topk_idx.cpu().numpy()):
        print(f"  {class_names[i]:10s} — {p*100:5.2f}%")

    return probs.cpu().numpy(), topk_idx.cpu().numpy()


In [ ]:
probs, idx = predict_outside_image("your_photo.png")